In [ ]:
import cv2
import numpy as np


def stain_features(image: np.ndarray, window: int | None = None, k: int | None = None) -> dict:
    """Measure how blotchy the paper background is.
    Returns a dict of scores used by is_stain()."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    h, w = gray.shape

    # 1. Paper background: dilate to erase ink / bleed-through, median blur to smooth
    if window is None:
        window = max(3, int(w * 0.04) | 1)
    if k is None:
        k = max(3, int(w * 0.006) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    background = cv2.medianBlur(cv2.dilate(gray, kernel), window)

    # Work on a small copy for speed
    scale = min(1.0, 300 / max(h, w))
    small = cv2.resize(background, None, fx=scale, fy=scale,
                       interpolation=cv2.INTER_AREA).astype(np.float32)
    sh, sw = small.shape

    # 2. Fit a smooth degree-2 surface (shadows / uneven lighting).
    #    Iteratively ignore dark pixels so stains don't pull the fit down.
    yy, xx = np.mgrid[0:sh, 0:sw]
    x, y, z = xx.ravel() / sw, yy.ravel() / sh, small.ravel()
    A = np.stack([x**i * y**j for i in range(3) for j in range(3 - i)], axis=1)
    keep = np.ones_like(z, bool)
    for _ in range(4):
        coef, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
        resid = z - A @ coef
        keep = resid > -2 * resid[keep].std()
    fit = (A @ coef).reshape(sh, sw)

    # 3. Residual relative to paper brightness: stains show up as negative blobs
    paper = max(np.median(fit), 1.0)
    rel = (small - fit) / paper
    blotchiness = float(rel.std())

    # 4. Stain mask: noticeably darker than the fitted paper
    stain_mask = (rel < -0.08).astype(np.uint8) * 255
    stain_mask = cv2.morphologyEx(stain_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    # 5. Keep only large blobs (stains are big, text residue is tiny)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(stain_mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    min_blob = 0.005 * sh * sw
    large = areas[areas >= min_blob]
    area_fraction = float(large.sum() / (sh * sw))

    # 6. Optional colour cue: stains are usually yellow/brown (higher LAB b*)
    yellowness = 0.0
    if image.ndim == 3 and len(large) > 0:
        colour_bg = cv2.medianBlur(cv2.dilate(image, kernel), window)
        lab = cv2.cvtColor(cv2.resize(colour_bg, (sw, sh)), cv2.COLOR_BGR2LAB)
        b = lab[:, :, 2].astype(np.float32)
        big_ids = 1 + np.where(areas >= min_blob)[0]
        in_stain = np.isin(labels, big_ids)
        if (~in_stain).any():
            yellowness = float(b[in_stain].mean() - b[~in_stain].mean())

    return {
        "blotchiness": blotchiness,
        "area_fraction": area_fraction,
        "num_large_blobs": int(len(large)),
        "yellowness": yellowness,
    }


def is_stain(image: np.ndarray,
             blotch_thresh: float = 0.03,
             area_thresh: float = 0.01) -> tuple[bool, dict]:
    """True if the image looks stained. Calibrate the thresholds on your own images."""
    f = stain_features(image)
    stained = f["blotchiness"] > blotch_thresh and f["area_fraction"] > area_thresh
    return stained, f


# ---------- Calibration: run on your labelled folders and compare the scores ----------
# from pathlib import Path
# for label in ["stain", "bleedthrough", "blur"]:
#     for p in sorted(Path(f"samples/{label}").glob("*")):
#         img = cv2.imread(str(p))
#         if img is None:
#             continue
#         stained, f = is_stain(img)
#         print(f"{label:13s} {p.name:30s} stain={stained}  "
#               f"blotch={f['blotchiness']:.4f}  area={f['area_fraction']:.4f}  "
#               f"blobs={f['num_large_blobs']}  yellow={f['yellowness']:.1f}")